In [0]:
import os

BASE_PATH = "/Volumes/dataplatform01_central_dev_catalog_01/bronze_raw_sap_bo/sapbo_webi/prod/object_type=csv/" 
# "/Volumes/dataplatform01_central_dev_catalog_01/bronze_raw_sap_bo/sapbo_webi/dev/object_type=csv/ingestion_date=2026-06-01/"

# --- Step 1: collect all CSV file paths ---
csv_files = []
for root, dirs, files in os.walk(BASE_PATH):
    # Skip hidden directories to avoid slowdowns/timeouts
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    for f in files:
        if f.upper().endswith(".CSV"):
            csv_files.append(os.path.join(root, f))

csv_files = sorted(csv_files)
print(f"Total CSV files found: {len(csv_files)}")

# --- Read raw content of each file ---
raw_files = {}   # filename -> list of raw lines
failed_files = []
for path in csv_files:
    filename = os.path.basename(path)
    try:
        with open(path, "r", encoding="utf-8", errors="replace") as fh:
            raw_files[filename] = fh.readlines()
    except Exception as e:
        failed_files.append((filename, str(e)))
        print(f"  WARNING: skipped {filename} — {e}")

print(f"Files read into memory: {len(raw_files)}")
if failed_files:
    print(f"Files failed to read: {len(failed_files)}")

# --- Build files inventory DataFrame ---
import re
import pandas as pd

records = []
for path in csv_files:
    filename = os.path.basename(path)
    match = re.search(r"(\d+)", filename)
    doc_id = match.group(1) if match else None
    records.append({"Report Type": "WEBI", "Report Document ID": doc_id})

df_files = pd.DataFrame(records)
print(f"\nFiles inventory ({len(df_files)} rows):")
# display(df_files)

In [0]:
# Matches field values that look like data (numbers, currencies, dates)
_DATA_LIKE = re.compile(r'[\u20ac$%]|^\d[\d,.\s]*$|^\d{2}[/\-]\d{2}')


def classify_section(lines):
    """
    Type 1 - Header + Data   : multi-line, first line = text column headers, rest = data rows.
    Type 2 - Data only       : multi-line, first line looks like data (numeric/date/empty
                               first field) -- continuation block with no header.
    Type 3 - Title / Text    : no tabs in any line -- report title, label, free text.
    Type 4 - Header, no data : single line, all tab-separated fields are text labels
                               (empty table header with zero data rows).
    Type 5 - Orphaned data   : single line with tabs containing at least one data value
                               (isolated row split off from a table by a blank line).
    """
    first = lines[0]
    all_no_tab = all('\t' not in line for line in lines)

    if all_no_tab:
        return 3  # pure text / title

    if len(lines) == 1:
        fields = [f.strip() for f in first.split('\t') if f.strip()]
        has_data = any(_DATA_LIKE.search(f) for f in fields)
        return 5 if has_data else 4

    # Multi-line with tabs: check whether first line is a header or data
    first_field = first.split('\t')[0].strip()
    is_data_first = (
        not first_field
        or bool(re.match(r'^[\d,.\-\s]+$', first_field))
        or bool(re.match(r'^\d{2}[/\-]\d{2}[/\-]\d{4}$', first_field))
    )
    return 2 if is_data_first else 1


SECTION_TYPE_LABEL = {
    1: "Header + Data",
    2: "Data only",
    3: "Title / Text",
    4: "Header, no data",
    5: "Orphaned data row",
}


def split_into_sections(lines):
    """Split raw file lines into sections separated by one or more blank lines."""
    sections = []
    current = []
    for line in lines:
        stripped = line.rstrip("\n").rstrip("\r")
        if stripped.strip() == "":
            if current:
                sections.append(current)
                current = []
        else:
            current.append(stripped)
    if current:
        sections.append(current)
    return sections


# Apply to every file
file_sections = {}   # filename -> list of sections (each section = list of lines)
for filename, lines in raw_files.items():
    file_sections[filename] = split_into_sections(lines)

total_sections = sum(len(s) for s in file_sections.values())
print(f"Total sections across all files: {total_sections}")

# --- Build sections DataFrame ---
records = []
for filename, sections in file_sections.items():
    match = re.search(r"(\d+)", filename)
    doc_id = match.group(1) if match else None
    total = len(sections)
    for idx, sec in enumerate(sections):
        stype = classify_section(sec)
        records.append({
            "Report Type":        "WEBI",
            "Report Document ID": doc_id,
            "Total Sections":     total,
            "Section Index":      idx,
            "Section Type":       stype,
            "Section Type Label": SECTION_TYPE_LABEL[stype],
            "Header Content":     sec[0].replace('\t', ' | ') if stype in (1, 4) else None,
        })

df_sections = pd.DataFrame(records)
print(f"\nSections DataFrame ({len(df_sections)} rows):")

type_dist = df_sections.groupby(["Section Type", "Section Type Label"]).size().reset_index(name="Count")
print("\nSection type distribution:")
print(type_dist.to_string(index=False))
df_sections.columns = [c.replace(" ", "_") for c in df_sections.columns]
spark_df_sections = spark.createDataFrame(df_sections)
spark_df_sections.write.format("delta").mode("overwrite").saveAsTable("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_output_headers")

In [0]:
%sql 
select roh1.Report_Document_ID, case when rohc1.Report_Document_ID is null then 'No columns found' else cast(count(distinct rohc1.Header_Column) as string)
 end as Result from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_output_headers as roh1
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_output_headers_columns as rohc1
on roh1.Report_Document_ID = rohc1.Report_Document_ID
group by roh1.Report_Document_ID, rohc1.Report_Document_ID

In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_SQL_Fields

select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_output_headers
where Report_Document_ID  in ('253113','329739','252204','8569849','253092','9619233','12339300','306274')

select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
where Document_Id in ('253113','329739','252204','8569849','253092','9619233','12339300','306274')
select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_output_headers_columns
where Report_Document_ID = '10224691'

In [0]:
import re
from pyspark.sql.functions import udf, col, explode, split, when, size, element_at, trim, upper,expr
from pyspark.sql.types import ArrayType, StringType
# from sqlglot.expressions import Table, CTE, Column    
    
df = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_output_headers")
df_headers = df.select(
    col("Report_Type"),
    col("Report_Document_ID"),
    explode(
        split(col("Header_Content"), r"\|")
    ).alias("Header_Column")
).withColumn("Header_Column", trim(col("Header_Column")))
# Replace spaces with underscores and remove non-alphanumeric/underscore characters
from pyspark.sql.functions import regexp_replace

df_headers = df_headers.withColumn(
    "Header_Column",
    regexp_replace(
        regexp_replace(col("Header_Column"), " ", "_"),
        r"[^A-Za-z0-9_]", ""
    )
)
df_headers=df_headers.dropDuplicates()
df_headers.write.format("delta").mode("overwrite").saveAsTable("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_output_headers_columns")